# Session 13 · Homework — Advising the Fraud Team

**Machine Learning Foundations · Sanketana School of Code**

You are advising a bank's fraud team on `fraud_transactions.csv`. This is the **capstone of the classification module**: model, threshold, metrics, and consequences in one recommendation.

## Step 1 · The confusion matrix at 0.5

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, precision_score, recall_score

fraud = pd.read_csv("../../../datasets/secondary/fraud_transactions.csv")
X = fraud.drop(columns=["is_fraud", "transaction_id"]).values
y = fraud["is_fraud"].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler().fit(X_train)
model = LogisticRegression(max_iter=1000).fit(scaler.transform(X_train), y_train)
proba = model.predict_proba(scaler.transform(X_test))[:, 1]

In [ ]:
def draw_matrix(y_true, pred, ax, title):
    """Confusion matrix, actual-fraud row first, cells named in plain language."""
    # labels=[1, 0] puts 'fraud' first on both axes, matching the board grid.
    cm = confusion_matrix(y_true, pred, labels=[1, 0])
    names = [["caught\n(TP)", "missed\n(FN)"],
             ["false alarm\n(FP)", "cleared\n(TN)"]]
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{names[i][j]}\n{cm[i, j]}", ha="center", va="center",
                    fontsize=11, color="black")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["fraud", "legit"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["fraud", "legit"])
    ax.set_xlabel("PREDICTED"); ax.set_ylabel("ACTUAL")
    ax.set_title(title)

In [ ]:
pred_05 = (proba >= 0.5).astype(int)
fig, ax = plt.subplots(figsize=(5, 5))
draw_matrix(y_test, pred_05, ax, "Fraud model at threshold = 0.5")
plt.tight_layout(); plt.show()
print("recall:", round(recall_score(y_test, pred_05), 2),
      " precision:", round(precision_score(y_test, pred_05), 2))

## Step 2 · Meet the policy: catch at least 80% of fraud

Find a threshold with **recall ≥ 0.80**, draw its confusion matrix, and report the extra false-alarm cost versus 0.5.

In [ ]:
# ✏️ TODO: find the highest threshold that still reaches 80% recall.
choice = None
for thr in np.round(np.arange(0.05, 0.95, 0.05), 2):
    if recall_score(y_test, (proba >= thr).astype(int)) >= 0.80:
        choice = thr

pred_choice = (proba >= choice).astype(int)
fig, ax = plt.subplots(figsize=(5, 5))
draw_matrix(y_test, pred_choice, ax, f"policy threshold = {choice}")
plt.tight_layout(); plt.show()

fp_choice = int(((pred_choice==1)&(y_test==0)).sum())
fp_05 = int(((pred_05==1)&(y_test==0)).sum())
print("policy threshold:", choice,
      " recall:", round(recall_score(y_test, pred_choice), 2),
      " precision:", round(precision_score(y_test, pred_choice), 2))
print("extra false alarms vs 0.5:", fp_choice - fp_05)

## Step 3 · ✏️ Your recommendation to the fraud team

Write **5–6 sentences**:

1. the **threshold** you recommend and its **recall & precision**;
2. what each of the **four cells** means as a real consequence;
3. **who benefits and who pays** at your setting;
4. the strongest **objection** the customer-experience team would raise.

*Your recommendation here:*

*Check your work against `homework-solutions.ipynb` after you've attempted every part.*